# A3. Numbers for Tables 2 and 4 (headline = pooled leave-one-year-out R²)

P3: numbers for Tables 1, 2 and 4. Writes CSVs to Paper/figures/data/ and a readable Paper/tables.md.

In [1]:
# Setup
import sys
from pathlib import Path
HERE = Path.cwd().resolve()          # notebooks/ folder
sys.path.insert(0, str(HERE.parent)); sys.path.insert(0, str(HERE.parents[1]))
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
from style import *   # PM25, O3, NAVY, GREY, RED, LIGHT, ROOT, DATA_OUT, save_fig, panel_label

In [2]:
# Analysis
loyo = pd.read_csv(DATA_OUT / "loyo_summary.csv").set_index("pollutant")
cmp_ = pd.read_csv(DATA_OUT / "equation_vs_persistence_common_rows.csv").set_index("pollutant")
r2 = pd.read_csv(DATA_OUT / "front_r2_recomputed.csv")
B = ROOT / "Data/processed/colombia/bogota"
fs = {"pm25": json.load(open(B / "bogota_forecast_feature_selection.json")), "o3": json.load(open(B / "bogota_o3_forecast_feature_selection.json"))}
pipe = {"pm25": json.load(open(B / "bogota_feature_selection.json"))["scores_no_lag"][-1],   # SFS + neural-net pipeline, pooled LOYO = last score of the SFS (2026-10-20: read from the re-run selections)
        "o3": json.load(open(B / "bogota_o3_feature_selection.json"))["scores_no_lag"][-1]}
def front_at(name, c):
    s = r2[(r2.front == name) & (r2.complexity <= c)].sort_values("complexity").iloc[-1]; return int(s.complexity), float(s.r2_insample)
rows = []
# 2026-10-06: O3 lead = dimensionally constrained front with T in the pool, complexity 15 (was free-form c16)
for pol, lead_front, lead_c, lag_front in [("pm25", "bogota_pm25_units", 13, "bogota_pm25_forecast"), ("o3", "bogota_o3_units_temp", 15, "bogota_o3_forecast")]:
    ic = r2[(r2.front == lead_front) & (r2.complexity == lead_c)].iloc[0]
    lc, lr = front_at(lag_front, lead_c)
    rows.append(dict(pollutant={"pm25": "PM2.5", "o3": "O3"}[pol], lead_complexity=lead_c, in_sample_r2=ic.r2_insample, n_rows=int(loyo.loc[pol, "n"]),
                     loyo_r2=loyo.loc[pol, "pooled_r2"], persistence_r2=cmp_.loc[pol, "persistence_pooled"], pipeline_nn_ceiling=pipe[pol],
                     SI_loyo_mean_of_years=loyo.loc[pol, "mean_of_years_r2"], SI_persistence_mean_of_years=cmp_.loc[pol, "persistence_mean_of_years"],
                     lagged_front_complexity=lc, lagged_front_in_sample_r2=lr,
                     lagged_feature_set_loyo_pooled=fs[pol]["no_self_lag_no_calendar"]["final_loyo_r2"]))
t2 = pd.DataFrame(rows); t2.to_csv(DATA_OUT / "table2_equations.csv", index=False)   # keeps the SI_ columns

att = pd.read_csv(DATA_OUT / "mechanism_attenuation_canonical.csv")
def verdict(r):
    # R2 (2026-09-22): use the bootstrap CI on the CHANGE in r, not a fixed % threshold -- matches Fig 6a's colour scheme.
    if r.control.startswith("none"): return "reference"
    if not r.attenuation_ci_excludes_zero: return "not distinguishable from no change (bootstrap CI includes 0)"
    return "accounts for part of the association" if r.pct_attenuation > 0 else "does not account for it (partial r rises)"
att["verdict"] = att.apply(verdict, axis=1)
att["partial_r_ci"] = att.apply(lambda r: f"{r.partial_r:.3f} ({r.ci_lo:.3f}, {r.ci_hi:.3f})", axis=1)
def _pm(v): return f"{-v:+.1f}".replace("-", "−")   # sign flipped to match Fig 6a: a fall in r prints as negative throughout
att["change_in_r_pct"] = att.apply(lambda r: f"{_pm(r.pct_attenuation)} ({_pm(r.pct_ci_hi)}, {_pm(r.pct_ci_lo)})" if r.control != "none (raw corr)" else "0.0 (ref.)", axis=1)
t4 = att[["control", "controls", "partial_r_ci", "change_in_r_pct", "n", "verdict"]]; t4.to_csv(DATA_OUT / "table4_mechanisms.csv", index=False)

t1 = pd.read_csv(DATA_OUT / "table1_data_inventory.csv")
def md(df): return df.to_markdown(index=False)
out = ["# Table numbers (generated by notebook A3_tables_2_and_4 -- do not hand-edit)\n",
       "## Table 1. Data inventory (period and completeness computed from the master datasets)\n",
       md(t1[["variable", "description", "source", "native_resolution", "units", "Bogota_first", "Bogota_completeness_pct", "Medellin_first", "Medellin_completeness_pct", "note"]].fillna("")),
       "\nCompleteness = valid days / days in the city's modelling window (Bogota 2014-01-01 to 2022-12-31, 3287 d; Medellin 2013-01-01 to 2022-12-31, 3652 d; R8, 2026-09-22 -- not the full download range).\n",
       "## Table 2. The two lead equations (Bogota)\n", md(t2[[c for c in t2.columns if not c.startswith("SI_")]].round(3)),
       "\nAll R2 are for daily means. `loyo_r2` is the R2 of the pooled out-of-fold predictions of leave-one-year-out cross-validation (constants refitted per fold; the same quantity the feature-selection pipeline uses). Persistence is a linear fit on yesterday's value of the same pollutant, evaluated on the same days and protocol. `in_sample_r2` is on all rows at once. The mean of per-year R2 and all other metrics are in the supplement (Paper/tables_SI.md). The lagged-only front R2 is in-sample at the largest complexity not exceeding the lead equation's; `lagged_feature_set_loyo_pooled` is the cross-validated (pooled) score of the lagged-only selected features.\n",
       "## Table 4. Mechanism tests on the canonical Bogota sample\n", md(t4),
       "\nPartial correlation of PM2.5 with O3 (residual-on-residual OLS with intercept); CIs are 95% moving-block-bootstrap intervals (30-day blocks, paired with the raw correlation; R2, 2026-09-22 -- supersedes the Fisher-z intervals of the first draft, which treated days as independent and were too narrow). Change in r is relative to the raw correlation, with its own paired bootstrap interval; a verdict of 'not distinguishable from no change' means that interval includes 0.\n"]
(ROOT / "Paper" / "tables.md").write_text("\n".join(out))
print(t2.round(3).T.to_string()); print(t4.to_string(index=False))

                                    0      1
pollutant                       PM2.5     O3
lead_complexity                    13     15
in_sample_r2                    0.684  0.609
n_rows                           3283   2917
loyo_r2                         0.677  0.603
persistence_r2                  0.569  0.585
pipeline_nn_ceiling             0.724  0.658
SI_loyo_mean_of_years           0.655  0.536
SI_persistence_mean_of_years    0.516  0.525
lagged_front_complexity            12     15
lagged_front_in_sample_r2       0.489   0.46
lagged_feature_set_loyo_pooled  0.493  0.492
                           control                         controls         partial_r_ci      change_in_r_pct    n                                                      verdict
                   none (raw corr)                              NaN 0.493 (0.401, 0.579)           0.0 (ref.) 2917                                                    reference
                      NO2 (ground)                             